
# race critial perterson mutex test and set, compare and swaping

<br>

**Question 1:** A university system generates students IDs using a shared variable next_id. When multile admission requests are processed simultaneously, some students receive duplicated IDs. No synchronization mechanism is currently used .

<br>

In [18]:

import time 
import threading

#shared_variable
next_id = 5 

def assign_ids(std_name:str)->None:
    global next_id 
    
    current_id = next_id
    print("Student name: {} receive id: {}".format(std_name,current_id))
    
    # sleep: (in this time other thread can read this the shared variables)
    time.sleep(1)
    
    next_id = current_id+1
    print("Assiged Id: {} Student: {} Next ID: {}".format(current_id,std_name,next_id))
    
    
# create thread object in memory 
# must be add , at the end python will understand it's a tuple
t1 = threading.Thread(target=assign_ids,args=("Student-A",))
t2 = threading.Thread(target=assign_ids,args=("Student-B",))

# start the threads:
t1.start()
t2.start()

# wait until thread does not finished his execution:
t1.join()
t2.join()


Student name: Student-A receive id: 5
Student name: Student-B receive id: 5
Assiged Id: 5 Student: Student-A Next ID: 6
Assiged Id: 5 Student: Student-B Next ID: 6


In [21]:


# 1. Mutex:
next_id = 5 

# create a lock object
lock = threading.Lock()

def assign_ids(std_name:str)->None:
    global next_id
    
    # aquire
    try:
        lock.acquire()
        current_id = next_id
        print(f"Student name: {std_name} receive id: {current_id}")
        
        # wait that other's thread try to access the shared variables
        time.sleep(1)
        
        next_id = current_id+1
        print(f"Assiged Id: {current_id} Student: {std_name} Next ID: {next_id}")
    finally:
        lock.release()
    
t1 = threading.Thread(target=assign_ids,args=("Std_A",))
t2 = threading.Thread(target=assign_ids,args=("Std_B",))

t1.start()
t2.start()
t1.join()
t2.join()

Student name: Std_A receive id: 5
Assiged Id: 5 Student: Std_A Next ID: 6
Student name: Std_B receive id: 6
Assiged Id: 6 Student: Std_B Next ID: 7


In [22]:


# 1. Mutex problem with python context manager with:
next_id = 5 
lock = threading.Lock()

def students_ids(std_name:str)->None:
    global next_id
    
    with lock:
        current = next_id
        print(f"{std_name} assign id: {current}")
        
        time.sleep(1)
        next_id = current+1 
        print(f"{std_name} assign id: {current} next_id: {next_id}")
        print("*"*30)
        

t1 = threading.Thread(target=students_ids,args=("STD_A",))
t2 = threading.Thread(target=students_ids,args=("STD_B",))

print("*"*30)

t1.start()
t2.start()
t1.join()
t2.join()



******************************
STD_A assign id: 5
STD_A assign id: 5 next_id: 6
******************************
STD_B assign id: 6
STD_B assign id: 6 next_id: 7
******************************


In [23]:

# 2. === Test and Set ===

next_id = 5 
# mean no one in critical section
target = False

def test_and_set()->bool:
    global target
    r = target
    target = True
    return r 


def students_ids(std_name:str):
    global next_id 
    
    # === Entry section: text_and_set() ===
    while test_and_set():
        pass 
    
    
    # === critical section: ===
    try: 
        current = next_id
        print(f"{std_name} assign id: {current}")
        
        
        time.sleep(1)
        
        next_id = current+1 
        print(f"{std_name} assign id: {current} next_id: {next_id}")
        print("*"*30)
    
    # === exit section ===
    finally:
        global target
        target = False 
        
    
t1 = threading.Thread(target=students_ids,args=("STD_A",))
t2 = threading.Thread(target=students_ids,args=("STD_B",))

t1.start()
t2.start()
t1.join()
t2.join()



STD_A assign id: 5
STD_A assign id: 5 next_id: 6
******************************
STD_B assign id: 6
STD_B assign id: 6 next_id: 7
******************************


In [ ]:


# === 3. peterson solution: ===
turn = 0 
next_id = 5 
interested = [False, False]

def entry_section(process:int):
    global turn 
    global interested
    
    other = 1- process
    interested[process] = True 
    turn = process
    
    while (interested[other]==True and turn==process):
        pass 
    

def exit_section(process:int):
    global interested
    interested[process] = False
    

def students_ids():
    global next_id
    pass 

    
